# GSB 5544 — PA 5.2: HTML and Web Scraping — INSTRUCTOR SOLUTION

**How to use this notebook.** Each part is answered in the same beats: **Approach** → **code** (complete and
executed) → **What the code does** → **Expected output** → **Common mistakes**.

**The one idea to keep returning to:** *solve it for one, then loop.* Parts 7 – 11 extract a single city by
hand precisely so that part 12 is a copy-and-paste into a `for` loop. The same rhythm repeats for the hockey
pages: one page first, then all of them.

**Live pages.** Wikipedia is edited continuously. The numbers here were correct when this notebook was executed
(September 2026). If the table's `class` attribute or its column order changes, parts 5 – 13 need the
corresponding edit — the *Inspect* step in part 4 is how you find out.

In [1]:
import pandas as pd

## Scraping an HTML table

Open the URL https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population and scroll down until you see a table of the cities in the U.S. with population over 100,000 (as of Jul 1, 2025). We'll use Beautiful Soup to scrape information from this table.

1\. Read in the HTML from the URL using the `requests` library.

### Solution 1 — get the HTML

**Approach.** `requests.get` downloads the page, exactly as it fetched JSON last time; the HTML is in
`response.text`. **Wikipedia rejects requests that do not identify themselves**, so send a `User-Agent` header.

In [2]:
import requests

url = "https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population"
headers = {"User-Agent": "GSB5544-class-exercise/1.0 (Cal Poly; educational use)"}

response = requests.get(url, headers=headers)
print(response.status_code)
print(len(response.text), "characters of HTML")
response.text[:300]

200
1661117 characters of HTML


'<!DOCTYPE html>\n<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disab'

**What the code does.**
- `requests.get(url, headers=headers)` sends an HTTP GET, just like a browser's address bar.
- `headers={"User-Agent": ...}` says who is asking. By default `requests` announces itself as
  `python-requests/2.x`, and Wikipedia's servers answer that with **403 Forbidden**. Any honest descriptive
  string works; Wikimedia's policy asks for one that identifies the project.
- `response.status_code` — 200 means the page arrived. `response.text` is the page source as one long string
  (about 1.7 million characters); printing a slice is enough to see that it is HTML.

**Expected output.** `200`, then roughly 1.7 million characters beginning `<!DOCTYPE html>`.

**Common mistakes.**
- **No header → 403.** The student's `response.text` is then a ~100-character error page, and every later
  step fails in a misleading way: part 3 finds **0 tables**, part 6 raises `IndexError: list index out of range`.
  When a student reports either symptom, ask for `response.status_code` first.
- `response.json()` → `JSONDecodeError`: this is a web page, not an API.
- Using `response` (the object) where the *text* is needed in part 2.

2\. Use Beautiful Soup to parse this string into a tree called `soup`

### Solution 2 — parse into a tree

**Approach.** `BeautifulSoup` converts the string into a searchable tree of tags.

In [3]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "html.parser")
print(type(soup))
soup.title.text

<class 'bs4.BeautifulSoup'>


'List of United States cities by population - Wikipedia'

**What the code does.**
- `BeautifulSoup(html_string, "html.parser")` — the first argument is the HTML **text**; the second names the
  parser (`"html.parser"` ships with Python, so nothing extra to install).
- The result, `soup`, represents the whole document. `soup.title` is a shortcut to the first `<title>` tag, and
  `.text` is the text inside it — a quick confirmation that we parsed the page we meant to.

**Expected output.** `<class 'bs4.BeautifulSoup'>` and `'List of United States cities by population - Wikipedia'`.

**Common mistakes.**
- `BeautifulSoup(response, ...)` → `TypeError` (it needs the string, `response.text`).
- `BeautifulSoup(url, ...)` — parses the *address* as if it were HTML; no error, but `soup` contains no tables.
- Omitting the parser argument works but prints a warning; name it.
- `from bs4 import beautifulsoup` — the class name is case-sensitive; and the package installs as
  `beautifulsoup4` but imports as `bs4`.

3\. Determine how many tables are in `soup`. (Hint: use `find_all("table")`.)

### Solution 3 — how many tables?

**Approach.** `find_all("table")` returns a list of every `<table>` tag; `len` counts them.

In [4]:
tables = soup.find_all("table")
len(tables)

10

**What the code does.** `find_all(tag_name)` searches the entire tree beneath `soup` and returns a list-like
`ResultSet`. Because it is a list, `len()`, indexing (`tables[2]`), and `for` loops all work.

**Expected output.** **10** tables (as executed). The page holds the main cities table plus a sidebar, a legend,
tables for Puerto Rico, census-designated places, cities formerly over 100,000, and so on.

**Common mistakes.**
- `soup.find("table")` returns only the **first** table (a single tag); `len()` of a tag counts its children,
  giving a meaningless number rather than an error.
- Getting 0 → the download failed (part 1, status 403).

4\. There are several tables included in `soup`, so we need to narrow it down. Go to the cities table Wikipedia page and "Inspect" it. What are the attributes (class, style) of this table?



### Solution 4 — the attributes of the cities table

**Approach.** In the browser: right-click inside the cities table → **Inspect**, then move up the highlighted
lines until the whole table is highlighted; read the opening `<table ...>` tag. The same information is
available in Python from each tag's `.attrs`:

In [5]:
for i, t in enumerate(tables):
    print(i, "| class:", " ".join(t.attrs.get("class", [])), "| style:", t.attrs.get("style"))

0 | class: sidebar nomobile nowraplinks | style: None
1 | class: wikitable | style: None
2 | class: sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center | style: text-align:right
3 | class: wikitable | style: text-align:right;
4 | class: wikitable | style: text-align:left
5 | class: sortable wikitable static-row-numbers sort-under col1left | style: text-align:right
6 | class: wikitable sortable sort-under | style: None
7 | class: wikitable sortable sort-under col1left col2center | style: text-align:right
8 | class: nowraplinks mw-collapsible mw-collapsed navbox-inner | style: border-spacing:0;background:transparent;color:inherit
9 | class: navbox-columns-table | style: border-spacing: 0px; text-align:left;width:auto; margin-left:auto; margin-right:auto;


**What the code does.**
- Every tag has an `.attrs` dictionary of its attributes. `.get("style")` returns `None` for tables without a
  `style` rather than raising a `KeyError`.
- `class` comes back as a **list** (`['sortable', 'wikitable', ...]`) because an HTML element can carry several
  classes separated by spaces; `" ".join(...)` prints them as they appear in the source.
- `enumerate` numbers the tables so we can refer to them by position.

**Expected output.** The cities table is the one with
`class="sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center"` and
`style="text-align:right"` (index 2 in the list as executed).

**Points to make in class.**
- *Inspect* shows the page **after** the browser's JavaScript has run; `requests` gets the **raw source**. On
  Wikipedia they can differ slightly (a sortable table gains `jquery-tablesorter` in the browser). If a class
  copied from Inspect fails to match, print `.attrs` as above and trust what Python received.
- Many tables share `wikitable`; it is the **combination** of classes (plus the style) that is unique.

**Common mistakes.** Inspecting a *cell* or a *row* and reporting its attributes instead of climbing up to the
`<table>` tag; copying the class string with a missing or extra space.

5\. You should find that the cities table on the Wikipedia page corresponds to the element

```
<table class="sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center" style="text-align:right">
```

How many tables in `soup` have these attributes?

### Solution 5 — how many tables have these attributes?

**Approach.** Give `find_all` an `attrs` dictionary; only tags whose attributes match are returned.

In [6]:
cities_attrs = {
    "class": "sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center",
    "style": "text-align:right",
}
len(soup.find_all("table", attrs=cities_attrs))

1

**What the code does.** `attrs={"class": "...", "style": "..."}` requires **both** to match. When the class
value contains several space-separated classes, Beautiful Soup compares it with the tag's full `class`
attribute as written — so the string must match exactly, in the same order.

**Expected output.** **1** — the attributes identify the cities table uniquely, which is what makes part 6 safe.

**Common mistakes.**
- A typo anywhere in the long class string → 0 matches, and then `[0]` in part 6 raises `IndexError`.
- Matching on `{"class": "wikitable"}` alone → 7 tables: a *single* class name matches any tag that has that
  class among others.
- Writing `class="..."` as a keyword argument → `SyntaxError`, because `class` is a reserved word in Python.
  That is why we use the `attrs` dictionary (Beautiful Soup also accepts `class_=`).

6\. There should only be 1 table of this type, so we just need to select it. The following code finds all tables with the desired attributes and then selects the first (only) one to store as `table`. (You just need to run this.)

In [7]:
table = soup.find_all("table",
                  attrs={
                      "class": "sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center",
                      "style": "text-align:right"}
                  )[0]

7\. Our goal is now to scrape the information in `table` to create a Pandas data frame with one row for each city and columns for:

- city
- state
- population (2025 estimate)
- 2020 land area (sq mi).

First, let's just see how to scrape the information for New York City. Starting from `table` create an object, called `city`, that contains the information just for New York City.

Hints: Inspect the source; what kind of tag represents each row? Find all tags of this type in `table` and select the first one that corresponds to a city. Note that the first 3 rows of the table are headers.

### Solution 7 — the row for New York City

**Approach.** In HTML a table row is a `<tr>` tag. Search **inside `table`** (not `soup`) for all rows, look at
the first few to see where the data starts, and take the first city.

In [8]:
rows_all = table.find_all("tr")
print(len(rows_all), "rows")

for i in range(5):                                   # peek: which rows are headers?
    cells = rows_all[i].find_all(["th", "td"])
    print(i, [c.text.strip()[:14] for c in cells][:6])

351 rows
0 ['Municipality', 'ST', '2025estimate', '2020census', 'Change', '2020 land area']
1 ['mi2', 'km2', '/ mi2', '/ km2']
2 []
3 ['New York[c]', 'NY', '8,584,629', '8,804,190', '−2.49%', '300.5']
4 ['Los Angeles', 'CA', '3,869,089', '3,898,747', '−0.76%', '469.5']


In [9]:
city = rows_all[3]             # rows 0-2 are header rows; row 3 is the first city
print(city.text.strip()[:80])

New York[c]
NY
8,584,6298,804,190−2.49%
300.5778.3
29,29811,312
40°40′N 73°56′W﻿


**What the code does.**
- `table.find_all("tr")` — searching from `table` restricts the search to that table. `soup.find_all("tr")`
  would return the rows of all 10 tables.
- The peek loop prints the first few cells of rows 0 – 4. `find_all(["th", "td"])` accepts a **list** of tag
  names, so it shows header cells and data cells alike.
- `rows_all[3]`: rows 0 and 1 are the two-level header (*Municipality, ST, 2025 estimate…* and the *mi² / km²*
  sub-header), row 2 is an empty spacer row, and row 3 is New York.

**Expected output.** 351 rows (3 header rows + 348 cities). `city` is a single `<tr>` tag whose text begins
`New York[c] NY 8,584,629 …`.

**Common mistakes.**
- `table.find_all("tr")[0]` → the header row; part 8 then fails with `IndexError`, because header rows contain
  `<th>` cells and **no `<td>`**.
- `table.find("tr")` — also the header row (`find` = first match).
- Off-by-one on the index (2 → the empty spacer row → `IndexError` in part 8). Printing the rows, as above,
  beats counting in the browser.

8\. Starting with `city` extract the city's name and store it as `name`.

Hints: Inspect the source; what tag represents the cells within a row? Find all tags of this type and extract the text corresponding to the cell with the city's name.

### Solution 8 — the city's name

**Approach.** Cells within a row are `<td>` tags. The name is in the first cell — but so is a footnote marker,
so go one level deeper to the `<a>` (link) tag, which holds only the name.

In [10]:
cells = city.find_all("td")
print(len(cells), "cells")
print(repr(cells[0].text))            # the whole cell: includes the footnote marker

name = cells[0].find("a").text        # the link inside the cell: just the name
name

10 cells
'New York[c]'


'New York'

**What the code does.**
- `city.find_all("td")` → the row's 10 data cells, in column order: 0 name · 1 state · 2 2025 estimate ·
  3 2020 census · 4 change · 5 land area mi² · 6 land area km² · 7 density /mi² · 8 density /km² · 9 location.
- `cells[0].text` gathers **all** text inside the cell: `'New York[c]'`. The `[c]` is a footnote link.
- `cells[0].find("a")` finds the first `<a>` tag inside that cell — the link to the city's article — whose
  text is exactly `'New York'`.
- `repr(...)` shows hidden characters (`\n`, non-breaking spaces) that `print` would hide — a good debugging habit
  when scraping.

**Expected output.** `'New York'`.

**Common mistakes.**
- Stopping at `cells[0].text` → 28 of the 348 cities keep a footnote marker (`New York[c]`, `Philadelphia[d]`,
  `Jacksonville[e]`, …). It looks fine in `.head()` apart from the first row and then breaks a later merge on city
  name. Alternatives if students did not think of the `<a>` tag: `.text.split("[")[0]`, or Week 4's
  `re.sub(r"\[.*\]", "", text)`.
- `city.find("td").text` works for the name (first cell) but does not generalise to parts 9 – 11.
- Forgetting `.text` and storing the **tag** (`<a href=...>New York</a>`) — it prints plausibly but is not a string.

9\. Extract the city's state and store it as `state`.

### Solution 9 — the state

**Approach.** Second cell. Plain text, with `.strip()` for safety.

In [11]:
state = cells[1].text.strip()
state

'NY'

**What the code does.** `cells[1]` is the *ST* column; `.text` extracts the text; `.strip()` removes any
surrounding whitespace or newline (Week 4). Wikipedia cells frequently end in `\n`.

**Expected output.** `'NY'` — the table gives two-letter postal abbreviations, not state names.

**Common mistakes.** Forgetting that `cells` was defined in part 8 and re-searching from `table` (which gives
the first cells of the *whole table*); skipping `.strip()` and later failing to match `"NY\n" == "NY"`.

10\. Extract the city's population and store is as `population`.

### Solution 10 — the population

**Approach.** Third cell — but scraped values are **text**. Remove the thousands separators, then convert.

In [12]:
print(repr(cells[2].text))

population = int(cells[2].text.strip().replace(",", ""))
population

'8,584,629'


8584629

**What the code does.** `cells[2].text` is the string `'8,584,629'`. `.replace(",", "")` deletes the commas
(→ `'8584629'`) and `int(...)` turns the digits into a number that can be summed, sorted, and plotted.

**Expected output.** `8584629` (the 2025 estimate, as executed).

**Common mistakes.**
- `int("8,584,629")` → `ValueError: invalid literal for int()`. The comma is the culprit.
- Leaving it as text. Nothing fails — until the data frame sorts populations **alphabetically**
  (`'999,999'` > `'8,584,629'`) or `.sum()` concatenates strings. Check `df.dtypes` in part 12.
- Taking `cells[3]` (the 2020 census) — the question asks for the 2025 estimate.

11\. Extract the city's area and store is as `area`.

### Solution 11 — the land area

**Approach.** The 2020 land area in **square miles** is the sixth cell (index 5; index 6 is km²). It has a
decimal point, so convert with `float`.

In [13]:
print([c.text.strip() for c in cells[:8]])       # count along the row to find the right index

area = float(cells[5].text.strip().replace(",", ""))
area

['New York[c]', 'NY', '8,584,629', '8,804,190', '−2.49%', '300.5', '778.3', '29,298']


300.5

**What the code does.** Printing the row's cells is the reliable way to find the index: name, state, 2025
estimate, 2020 census, change, **land area mi²**, land area km², density… `float` rather than `int` because of
the decimal; `.replace(",", "")` because large areas are written like `1,706.8` (Anchorage).

**Expected output.** `300.5`.

**Common mistakes.**
- Counting columns from the **visible header**, where "2020 land area" is one header spanning two cells
  (mi² and km²) — students land on the wrong index. Count `<td>` cells, not header labels.
- `int("300.5")` → `ValueError`.
- Omitting the comma removal: it works for New York and then fails inside the loop at the first city with an
  area over 1,000 sq mi. A reminder that "works for one row" is a necessary test, not a sufficient one.

12\. Now put the steps for a single city into a loop to extract the information for all cities in `table` and create a data frame.

Hints:
- Start with an empty list named `rows`
- Write a loop that starts `for city in ...` and replace `...` code that finds all the table rows. (Use what you did in part 7, but don't just select one row. Select all rows except for the 3 header rows.)
- Use your code from 8-11 to extract the information for the city
- And append it to `rows` as "name", "state", "population", "area"
- Convert `rows` into a Pandas data frame. You should obtain a data frame with 348 rows and 4 columns.


### Solution 12 — every city: the loop

**Approach.** Parts 7 – 11 *are* the loop body. Replace "row 3" with "each row from 3 onward", collect one
dictionary per city in a list, and hand the list to `pd.DataFrame`.

In [14]:
rows = []
for city in table.find_all("tr")[3:]:                 # every row after the 3 header rows
    cells = city.find_all("td")
    rows.append({
        "name": cells[0].find("a").text,
        "state": cells[1].text.strip(),
        "population": int(cells[2].text.strip().replace(",", "")),
        "area": float(cells[5].text.strip().replace(",", "")),
    })

df_cities = pd.DataFrame(rows)
print(df_cities.shape)
df_cities.head()

(348, 4)


,name,state,population,area
0,New York,NY,8584629,300.5
1,Los Angeles,CA,3869089,469.5
2,Chicago,IL,2731585,227.7
3,Houston,TX,2397315,640.4
4,Phoenix,AZ,1665481,518.0


In [15]:
# Always audit a scrape: types, missing values, and the two ends of the table
print(df_cities.dtypes)
print("missing values:", df_cities.isna().sum().sum(), "| footnote markers left in names:",
      df_cities["name"].str.contains(r"\[").sum())
df_cities.tail(3)

name              str
state             str
population      int64
area          float64
dtype: object
missing values: 0 | footnote markers left in names: 0


,name,state,population,area
345,Davenport,IA,100358,63.8
346,Deltona,FL,100267,37.3
347,Longmont,CO,100109,28.8


**What the code does.**
- `[3:]` slices off the three header rows — the loop version of choosing `[3]` in part 7.
- The loop variable is called `city` on purpose: the lines inside are the code from parts 8 – 11, unchanged.
- `rows.append({...})` adds one dictionary per city. A **list of dictionaries** converts directly to a data
  frame: the keys become the column names.
- The audit cell confirms numeric dtypes (`int64`, `float64`), no missing values, and no `[` left in any name.

**Expected output.** A data frame with **348 rows and 4 columns**, New York / Los Angeles / Chicago at the top,
cities of just over 100,000 at the bottom.

**Common mistakes.**
- `[0:]` or no slice → `IndexError: list index out of range` on the very first pass (header rows have no `<td>`).
  A defensive alternative some students find: `if len(cells) == 0: continue`.
- `rows = []` placed **inside** the loop → a data frame with one row (the last city).
- `pd.DataFrame(...)` called inside the loop — slow, and only the last one survives.
- Appending a plain list `[name, state, population, area]` is fine, but then the columns are `0, 1, 2, 3` unless
  `columns=[...]` is given.
- Re-using the single-city variables (`name`, `state`, …) from parts 8 – 11 in the dictionary instead of
  recomputing them from the current `cells` → 348 copies of New York.
- 349 or 350 rows → a header row slipped in; 0 rows → `soup.find_all` was used and a different table was looped.

13\. Use the Pandas command `pd.read_html` can be used to scrape the table from the webpage. Note: `read_html` will return all the tables, so you will need to narrow your request using attributes. You don't need to worry about selecting columns; just scrape the whole table.

### Solution 13 — the same table with `pd.read_html`

**Approach.** `pd.read_html` parses `<table>` tags into data frames for you. It returns a **list** (one data
frame per table found), so narrow it with the same `attrs` and take element `[0]`. Pass it the HTML we already
downloaded, wrapped in `StringIO`.

In [16]:
from io import StringIO

tables_pd = pd.read_html(StringIO(response.text), attrs=cities_attrs)
print(len(tables_pd), "table(s) matched")

df_cities_pd = tables_pd[0]
print(df_cities_pd.shape)
df_cities_pd.head(3)

1 table(s) matched
(349, 10)


Municipality   ST 2025 estimate 2020 census  Change 2020 land area          \
  Municipality   ST 2025 estimate 2020 census  Change            mi2     km2   
0          NaN  NaN           NaN         NaN     NaN            NaN     NaN   
1  New York[c]   NY     8584629.0   8804190.0  −2.49%          300.5   778.3   
2  Los Angeles   CA     3869089.0   3898747.0  −0.76%          469.5  1216.0   

  2020 density                                        Location  
         / mi2    / km2                               Location  
0          NaN      NaN                                    NaN  
1      29298.0  11312.0    40°40′N 73°56′W﻿ / ﻿40.66°N 73.94°W  
2       8304.0   3206.0  34°01′N 118°25′W﻿ / ﻿34.02°N 118.41°W

In [17]:
# Optional tidy-up: flatten the two-level header and drop the empty spacer row
tidy = df_cities_pd.copy()
tidy.columns = [top if top == bottom else f"{top} {bottom}" for top, bottom in tidy.columns]
tidy = tidy.dropna(how="all").reset_index(drop=True)
print(tidy.shape)
tidy[["Municipality", "ST", "2025 estimate", "2020 land area mi2"]].head(3)

(348, 10)


,Municipality,ST,2025 estimate,2020 land area mi2
0,New York[c],NY,8584629.0,300.5
1,Los Angeles,CA,3869089.0,469.5
2,Chicago,IL,2731585.0,227.7


**What the code does.**
- `StringIO(response.text)` makes the string look like a file. Recent versions of pandas want HTML text passed
  this way (a bare string triggers a deprecation warning or is treated as a path).
- `attrs=cities_attrs` — the same dictionary as part 5 — so only the cities table is parsed. `match="Municipality"`
  (keep tables whose text contains that word) is an alternative.
- `[0]` takes the single data frame out of the list.
- The table has a two-row header, so pandas builds **MultiIndex columns** such as
  `('2020 land area', 'mi2')`. The tidy-up joins the two levels into one name.

**Expected output.** 1 table matched; shape **(349, 10)** — all ten columns, with numbers already converted. The
extra row (349 vs 348) is the empty spacer row, read as all-`NaN`; after `dropna(how="all")` it is 348.

**Points to make in class.** `read_html` is three lines instead of thirty — *when the data is a `<table>`*. But it
returns only the visible text: the names still carry their footnote markers (`New York[c]`), and links (`href`)
are lost. Beautiful Soup is for when you need something `read_html` does not give you, or when the data is not in
a table at all (the countries page in Topic 5.2).

**Common mistakes.**
- `pd.read_html(url)` directly → **`HTTPError: 403 Forbidden`**: pandas fetches the page itself *without* our
  `User-Agent` header. Download with `requests`, then pass the text.
- Forgetting that the result is a list: `tables_pd.head()` → `AttributeError: 'list' object has no attribute 'head'`.
- No `attrs`/`match` → a list of 10 data frames and guesswork about which index is right.
- `ImportError: lxml not found` on a fresh install — `pip install lxml` (or pass `flavor="bs4"`).

## Scraping from multiple webpages

We will scrape the hockey statistics from this website: https://www.scrapethissite.com/pages/forms/. Notice that the information is spread over many pages.

1\. Scrape the information from the first page with Beatiful Soup.

### Solution (hockey 1) — the first page

**Approach.** Exactly parts 1 – 2 again: request, check, parse. This site is built for scraping practice, so no
special header is needed (sending one does no harm).

In [18]:
base_url = "https://www.scrapethissite.com"

response = requests.get(base_url + "/pages/forms/")
print(response.status_code)

soup = BeautifulSoup(response.text, "html.parser")
soup.title.text.strip()

200


'Hockey Teams: Forms, Searching and Pagination | Scrape This Site | A public sandbox for learning web scraping'

**What the code does.** Same two steps as before. The site's address is kept in `base_url` because the page links
collected later are *relative* (`/pages/forms/?page_num=2`) and will need it in front. The variable is named
`soup` again because the activity's pagination cell further down expects that name.

**Expected output.** `200` and the title *Hockey Teams: Forms, Searching and Pagination | Scrape This Site …*.

**Common mistakes.** Re-using the Wikipedia `url` variable by accident; forgetting `.text`.

2\. Find the main table on this page and store it as `table`.

### Solution (hockey 2) — the main table

**Approach.** Count the tables first. If there is only one, `find` is all we need.

In [19]:
print(len(soup.find_all("table")), "table on the page")

table = soup.find("table")
table.attrs

1 table on the page


{'class': ['table']}

**What the code does.** `find_all` confirms the page has a single table; `find("table")` returns it as one tag.
Its only attribute is `class="table"`, so `soup.find("table", attrs={"class": "table"})` is an equivalent, more
explicit, way to write it.

**Expected output.** 1 table; `{'class': ['table']}`.

**Common mistakes.** `table = soup.find_all("table")` (no `[0]`) → `table` is a *list*, and the next step's
`table.find_all("tr")` raises `AttributeError: ResultSet object has no attribute 'find_all'` — Beautiful Soup's
message even suggests you probably treated a list of elements like a single element.

3\. Extract the information from the cells of this table into a Pandas data frame.

### Solution (hockey 3) — the table as a data frame

**Approach.** Inspect a row: team rows are `<tr class="team">` and the header row has no class — so asking for
`class="team"` rows skips the header automatically. Column names come from the `<th>` cells. Solve one row,
then loop.

In [20]:
header = [th.text.strip() for th in table.find_all("th")]
print(header)

team_rows = table.find_all("tr", attrs={"class": "team"})
print(len(team_rows), "team rows on this page")
[td.text.strip() for td in team_rows[0].find_all("td")]            # one row first

['Team Name', 'Year', 'Wins', 'Losses', 'OT Losses', 'Win %', 'Goals For (GF)', 'Goals Against (GA)', '+ / -']
25 team rows on this page


['Boston Bruins', '1990', '44', '24', '', '0.55', '299', '264', '35']

In [21]:
records = []
for team in team_rows:
    records.append([td.text.strip() for td in team.find_all("td")])

df_page1 = pd.DataFrame(records, columns=header)
df_page1.head()

,Team Name,Year,Wins,Losses,OT Losses,Win %,Goals For (GF),Goals Against (GA),+ / -
0,Boston Bruins,1990,44,24,,0.55,299,264,35
1,Buffalo Sabres,1990,31,30,,0.388,292,278,14
2,Calgary Flames,1990,46,26,,0.575,344,263,81
3,Chicago Blackhawks,1990,49,23,,0.613,284,211,73
4,Detroit Red Wings,1990,34,38,,0.425,273,298,-25


**What the code does.**
- `[th.text.strip() for th in ...]` is a **list comprehension** — a one-line loop that builds a list. Here it
  collects the nine header labels.
- `find_all("tr", attrs={"class": "team"})` keeps only the data rows. This is sturdier than slicing `[1:]`,
  because it selects rows by what they *are* rather than where they sit — which matters in part 4.
- **`.strip()` is essential on this site**: each cell's text is surrounded by newlines and indentation
  (`'\n        Boston Bruins\n    '`).
- Each row becomes a list of nine strings; `columns=header` names them.

**Expected output.** 25 rows × 9 columns: *Team Name, Year, Wins, Losses, OT Losses, Win %, Goals For (GF), Goals
Against (GA), + / -*. First row: Boston Bruins, 1990, 44 wins, 24 losses.

**Common mistakes.**
- No `.strip()` → every value wrapped in whitespace; later `df["Team Name"] == "Boston Bruins"` matches nothing.
- Looping over all `<tr>` without skipping the header → one row of `None`/empty values, or a column-count
  mismatch error.
- All columns are still **text** at this point. Conversion is done once, after all pages are collected (next part).
- The `OT Losses` column is **empty** for early seasons (overtime losses were not recorded before 1999–2000), so
  `int(...)` inside the loop raises `ValueError: invalid literal for int() with base 10: ''`. Convert afterwards with
  `pd.to_numeric(..., errors="coerce")`.

4\. But this only represents the first page of data. There are many pages of data. How do we scrape all of the data?

We could switch to different pages by modifying the `page_num` parameter in the URL.

Alternatively, we can just grab the links at the bottom of the page.

In [22]:
pagination = soup.find("ul", attrs={"class": "pagination"})
links = pagination.find_all("a")

Let's take a look at the links found.

In [23]:
for link in links:
  print(link.attrs["href"])

/pages/forms/?page_num=1
/pages/forms/?page_num=2
/pages/forms/?page_num=3
/pages/forms/?page_num=4
/pages/forms/?page_num=5
/pages/forms/?page_num=6
/pages/forms/?page_num=7
/pages/forms/?page_num=8
/pages/forms/?page_num=9
/pages/forms/?page_num=10
/pages/forms/?page_num=11
/pages/forms/?page_num=12
/pages/forms/?page_num=13
/pages/forms/?page_num=14
/pages/forms/?page_num=15
/pages/forms/?page_num=16
/pages/forms/?page_num=17
/pages/forms/?page_num=18
/pages/forms/?page_num=19
/pages/forms/?page_num=20
/pages/forms/?page_num=21
/pages/forms/?page_num=22
/pages/forms/?page_num=23
/pages/forms/?page_num=24
/pages/forms/?page_num=1


Now we can loop over `links` to make a request to the url for each page and scrape the data into a table similar to what we did for the first page. Write such a loop to extract the data and create a Pandas data frame.


A few technicalities:

- You might need to skip the "previous" and "next" buttons
- So you don't keep repeating headers, you will want to skip rows that don't represent teams.

### Solution (hockey 4) — every page

**Approach.** Wrap the page-1 work in a loop over the pagination links collected above: build the full URL →
request → parse → extract the team rows → pause. Collect everything in one list and build the data frame once.

In [24]:
for link in links[:2] + links[-2:]:                    # what do the links look like, first and last?
    print(repr(link.text.strip()), link.attrs["href"], "| aria-label:", link.attrs.get("aria-label"))

'1' /pages/forms/?page_num=1 | aria-label: None
'2' /pages/forms/?page_num=2 | aria-label: None
'24' /pages/forms/?page_num=24 | aria-label: None
'»' /pages/forms/?page_num=1 | aria-label: Next


In [25]:
import time

# REQUEST + EXTRACT — 24 pages with a half-second pause
records = []
for link in links:
    if link.attrs.get("aria-label") in ("Next", "Previous"):      # arrow buttons repeat a page we already have
        continue
    page_url = base_url + link.attrs["href"]                      # relative link -> full address
    page_soup = BeautifulSoup(requests.get(page_url).text, "html.parser")

    for team in page_soup.find("table").find_all("tr", attrs={"class": "team"}):   # team rows only -> no headers
        records.append([td.text.strip() for td in team.find_all("td")])

    time.sleep(0.5)

len(records)

582

In [26]:
# PROCESSING — name the columns, convert the numbers, audit
columns = ["team", "year", "wins", "losses", "ot_losses", "win_pct", "goals_for", "goals_against", "goal_diff"]
df_hockey = pd.DataFrame(records, columns=columns)

numeric = columns[1:]
df_hockey[numeric] = df_hockey[numeric].apply(pd.to_numeric, errors="coerce")     # '' -> NaN instead of an error

print(df_hockey.shape, "| duplicated rows:", df_hockey.duplicated().sum())
print(df_hockey["year"].min(), "to", df_hockey["year"].max(), "|", df_hockey["team"].nunique(), "teams")
print(df_hockey.dtypes)
df_hockey.tail()

(582, 9) | duplicated rows: 0
1990 to 2011 | 35 teams
team                 str
year               int64
wins               int64
losses             int64
ot_losses        float64
win_pct          float64
goals_for          int64
goals_against      int64
goal_diff          int64
dtype: object


,team,year,wins,losses,ot_losses,win_pct,goals_for,goals_against,goal_diff
577,Tampa Bay Lightning,2011,38,36,8.0,0.463,235,281,-46
578,Toronto Maple Leafs,2011,35,37,10.0,0.427,231,264,-33
579,Vancouver Canucks,2011,51,22,9.0,0.622,249,198,51
580,Washington Capitals,2011,42,32,8.0,0.512,222,230,-8
581,Winnipeg Jets,2011,37,35,10.0,0.451,225,246,-21


**What the code does.**
- **Skipping the arrows.** The last link is the "»" (*Next*) button; it carries `aria-label="Next"` and points to
  a page that is already in the list. Without the `continue`, that page is scraped twice. (On page 1 there is no
  "Previous" button, but the test covers it for any starting page.) Alternatives students may find: keep only
  links whose text `.isdigit()`, or `links[:-1]`.
- **`base_url + href`.** The hrefs are relative (`/pages/forms/?page_num=2`); `requests` needs the full address.
- **A different variable, `page_soup`, inside the loop** so the original `soup` (and `links`) are not overwritten
  mid-loop.
- **`class="team"`** filters out each page's header row — the "don't keep repeating headers" technicality —
  without any index arithmetic.
- **One list for all pages**, one `pd.DataFrame` call after the loop.
- **`pd.to_numeric(errors="coerce")`** converts every numeric column at once and turns the blank `OT Losses`
  entries into `NaN` rather than failing.
- The audit line checks the result from three angles: size, duplicates, and the range of years.

**Expected output.** **582 rows × 9 columns**, 0 duplicated rows, seasons **1990 – 2011**, 35 distinct team names,
numeric dtypes (`ot_losses` is `float64` because it contains `NaN`s — 224 of them, all before the 1999 season).

**Common mistakes.**
- **Not skipping "Next"** → 607 rows, 25 of them duplicates. `df.duplicated().sum()` catches it.
- **`requests.get(link.attrs["href"])`** without the base → `MissingSchema: Invalid URL '/pages/forms/?page_num=2'`.
- **Passing the tag instead of its href**: `requests.get(link)` → the same error, less obviously.
- **Overwriting `soup` inside the loop**, then re-running the pagination cell and getting links from page 24.
- **Header rows in the data** (from `find_all("tr")` without the class filter): a row whose "team" is
  `Team Name`, and numeric conversion then fails or coerces a whole row to `NaN`.
- **`int()` inside the loop** dying on the first empty `OT Losses` cell.
- **No `time.sleep`**: this practice site tolerates it, but the habit is what gets students blocked elsewhere.

**The other route the activity mentions.** Instead of harvesting links, generate the URLs from the pattern:
`for page_num in range(1, 25): requests.get(base_url + "/pages/forms/", params={"page_num": page_num})`. It needs
the number of pages in advance (24 — read it off the last numbered link), whereas following the links discovers
it. The site also accepts `per_page=100`, which cuts 24 requests to 6 — a nice illustration that reading the URL
a site uses can save most of the work.